# Talk to the People — Colab Setup

Run this notebook in **Google Colab** (Runtime → Change runtime type → GPU) or in **Cursor** with a Python kernel.

Uses **Hugging Face Qwen2.5-3B-Instruct** for local inference. Enable GPU for faster embeddings and generation.

## 1. Check environment

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")
try:
    import torch
    print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
except ImportError:
    print("PyTorch not yet installed")

!nvidia-smi

: 

: 

## 2. Mount Google Drive (Colab only)

Mount Drive to persist eval results. Results will be saved under `My Drive/Talk-to-the-People/eval/results/`.

In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    EVAL_OUTPUT_DIR = "/content/drive/MyDrive/Talk-to-the-People/eval/results"
    import os
    os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)
    print(f"Eval results will be saved to: {EVAL_OUTPUT_DIR}")
else:
    EVAL_OUTPUT_DIR = "eval/results"

Mounted at /content/drive
Eval results will be saved to: /content/drive/MyDrive/Talk-to-the-People/eval/results


## 3. Install dependencies

In [3]:
# Colab: clone or pull latest, then install from requirements.txt
import os
REPO_DIR = "/content/Talk-to-the-People-A-Grounded-Persona-Conversation-System"
RE_CLONE = True  # Set True to remove and re-clone (for major updates)

if IN_COLAB:
    if os.path.exists(REPO_DIR):
        if RE_CLONE:
            !rm -rf {REPO_DIR}
            !git clone https://github.com/Jennt54321/Talk-to-the-People-A-Grounded-Persona-Conversation-System.git
        else:
            %cd {REPO_DIR}
            !git pull origin main
            %cd /content
    else:
        !git clone https://github.com/Jennt54321/Talk-to-the-People-A-Grounded-Persona-Conversation-System.git
    %pip install -q -r {REPO_DIR}/requirements.txt
else:
    %pip install -q -r requirements.txt

Cloning into 'Talk-to-the-People-A-Grounded-Persona-Conversation-System'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 180 (delta 74), reused 157 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 22.53 MiB | 14.73 MiB/s, done.
Resolving deltas: 100% (74/74), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 125.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.4 MB/s eta 0:00:00


## 4. Project setup

**In Colab:** Project is cloned from GitHub. `%cd` into project root.

**In Cursor:** The notebook is already in the project; `PROJECT_DIR` will auto-detect.

In [4]:
from pathlib import Path

PROJECT_DIR = Path("/content/Talk-to-the-People-A-Grounded-Persona-Conversation-System")

if IN_COLAB:
    if not PROJECT_DIR.exists():
        !git clone https://github.com/Jennt54321/Talk-to-the-People-A-Grounded-Persona-Conversation-System.git
    get_ipython().run_line_magic('cd', str(PROJECT_DIR))
else:
    # Cursor / local: use cwd (run notebook from project root)
    PROJECT_DIR = Path.cwd()
    assert (PROJECT_DIR / "app" / "main.py").exists(), f"Project root not found. cd to project root first."

print(f"Project root: {PROJECT_DIR}")
sys.path.insert(0, str(PROJECT_DIR))

/content/Talk-to-the-People-A-Grounded-Persona-Conversation-System
Project root: /content/Talk-to-the-People-A-Grounded-Persona-Conversation-System


## 7. Run Eval

Execute the evaluation pipeline (retrieval -> generation -> A/B/C metrics).  
Results are saved to Google Drive when mounted (see step 2).

**This runs the full pipeline from scratch** (retrieval → generation → metrics). No cached retrieval or results are used; use `--rerun-retrieval` so Stage 1 always runs.

**Speed tips:** Set `EVAL_LIMIT` (e.g. `10`) to run fewer questions; reduce `EVAL_TOP_K` (e.g. `4`) for fewer chunks per question; use `--no-rerank` for faster retrieval (lower quality).

In [5]:
import os
from pathlib import Path
os.chdir(PROJECT_DIR)

# Run full pipeline from scratch (no cached retrieval or results)
QUESTIONS_FILE = "eval/questions_life.json"
!python eval/run_eval.py -q {QUESTIONS_FILE} -o "{EVAL_OUTPUT_DIR}" --rerun-retrieval

2026-02-25 15:10:10.907975: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772032210.941785    2240 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772032210.952996    2240 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772032210.974682    2240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772032210.974712    2240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772032210.974716    2240 computation_placer.cc:177] computation placer alr

## 8. Start the web app

In [5]:
import os
import subprocess
os.chdir(PROJECT_DIR)

if IN_COLAB:
    # Colab: run in background so you can run ngrok in the next cell
    subprocess.Popen(
        ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=str(PROJECT_DIR),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    import time
    time.sleep(3)
    print("App starting. Run the ngrok cell below to get a public URL.")
else:
    # Cursor / local: run in foreground (interrupt to stop)
    !uvicorn app.main:app --reload --port 8000

App starting. Run the ngrok cell below to get a public URL.


## 9. (Colab only) Expose with ngrok

Run in a **new cell** after the app has started to get a public URL:

In [7]:
if IN_COLAB:
    !pip install -q pyngrok
    from pyngrok import ngrok
    
    # Replace with your token from https://dashboard.ngrok.com/get-started/your-authtoken
    ngrok.set_auth_token("3A4hBkuBhaAG8cLaGPfzSC1irYq_7d5gDmimjZ3j2ho2qbXL7")
    
    public_url = ngrok.connect(8000)
    print(f"Open in browser: {public_url}")
else:
    print("Use http://localhost:8000")

Open in browser: NgrokTunnel: "https://dallyingly-laziest-alejandro.ngrok-free.dev" -> "http://localhost:8000"


In [8]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

True
Tesla T4
